<!-- paper-paired-role-banner -->
> 🟦 **실습 코드 · 왼쪽 창** — TODO를 먼저 직접 구현하세요. 막히면 오른쪽 정답의 같은 셀에서 필요한 한 줄만 확인합니다.

# 15. GCN — Semi-Supervised Classification with Graph Convolutional Networks

- **원 논문:** [Semi-Supervised Classification with Graph Convolutional Networks](https://arxiv.org/abs/1609.02907)
- **저자 / 발표:** Thomas N. Kipf and Max Welling · ICLR (2017)
- **난이도 / 예상 시간:** 중급 · 약 55분
- **선수 지식:** PyTorch 기본, 행렬곱, 그래프 adjacency/degree, cross entropy

이 노트북은 논문 결과 수치를 그대로 재현하는 대규모 benchmark가 아니라,
핵심 수식과 구조가 실제로 어떻게 동작하는지 확인하는 **하드웨어 인식 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하며, 작은 NumPy·Python 실험은
전송 비용을 피하기 위해 CPU에서 실행될 수 있습니다.

## 재현 범위와 완료 기준

**이 노트북이 재현하는 것**

두 community의 작은 합성 그래프에서 self-loop와 대칭 degree normalization을 직접 구현하고, 노드 네 개의 label만으로 2-layer GCN이 나머지 노드를 분류하는지 확인한다.

**원 논문과 다른 것**

논문은 Cora, Citeseer, Pubmed 등의 citation graph에서 sparse 연산을 사용한다. 여기서는 24-node dense adjacency를 사용하지만 Eq. (2), Eq. (8)–(10)의 전파와 masked loss는 같다.

완료하려면 TODO를 구현하고, 검증 셀의 `assert`를 모두 통과시키고, 마지막 관찰 질문에
자신의 말로 답하세요. 수치가 논문과 다르더라도 핵심 불변식과 비교 방향이 맞으면 성공입니다.

## 논문 ↔ 노트북 지도

        절 번호와 수식 번호는 위 링크의 대표 공개본을 기준으로 합니다. 판본에 따라 페이지는 달라질 수
        있으므로 **절 제목과 수식 번호**를 함께 사용하세요.

        | 원 논문의 위치 | 이 노트북의 대응 실습 | 확인할 증거 |
        |---|---|---|
        | §2, Eq. (2) | `GraphConvolution.forward`의 `A_hat @ X @ W` 전파 | node 수를 보존하는 layer 출력 shape와 finite gradient를 검사 |
| §2.2, Eq. (7)–(8) | `normalize_adjacency`의 self-loop와 대칭 degree normalization | 정규화 행렬의 shape·대칭성·유한값을 검사 |
| §3.1, Eq. (9)–(10) | `TinyGCN`의 2-layer propagation과 train mask cross entropy | 미공개 test node 정확도와 학습 loss 감소를 검사 |

        > 읽기 순서: 먼저 해당 절을 훑고 → 코드를 예측해 작성하고 → 출력이 논문의 주장과 왜
        > 연결되는지 한 문장으로 설명합니다.

## 핵심 수식과 구현 설계

$$
H^{(l+1)}=\sigma\!\left(
\hat D^{-1/2}\hat A\hat D^{-1/2}H^{(l)}W^{(l)}\right),
\qquad \hat A=A+I
$$

- $\hat A$는 self-loop를 추가한 adjacency, $\hat D$는 degree matrix입니다.
- symmetric normalization은 이웃 수가 다른 node의 message 크기를 보정합니다.
- adjacency 준비, normalization, graph convolution layer, node loss를 직접 분리합니다.
- $H^{(l)}:[N,D_l]$는 normalized adjacency $[N,N]$와 가중치 $[D_l,D_{l+1}]$를 거칩니다.

## 구현 선택 이유

- **원문에서 보존한 부분:** §2, Eq. (2)
- **노트북 구현:** `GraphConvolution.forward`의 `A_hat @ X @ W` 전파
- **이렇게 구현한 이유:** 완성된 고수준 model을 한 줄로 호출하면
  논문의 핵심 shape·mask·loss·상태 변화가 숨겨집니다. 따라서 이 계산을
  이름이 드러나는 class와 함수로 나누고 중간 tensor를 assertion으로
  검사합니다.
- **실패를 잡는 증거:** node 수를 보존하는 layer 출력 shape와 finite gradient를 검사를 출력 크기만 아니라
  구현 불변식으로 사용합니다.
- **축소 선택:** 논문은 Cora, Citeseer, Pubmed 등의 citation graph에서 sparse 연산을 사용한다. 여기서는 24-node dense adjacency를 사용하지만 Eq. (2), Eq. (8)–(10)의 전파와 masked loss는 같다.는 기본 실행에서 생략합니다.
  이는 Windows CPU에서도 반복 실험이 가능하게 하면서 핵심 메커니즘은
  유지하기 위한 선택입니다.

## 1. Citation graph를 두 community로 축소하기

논문 §2 Eq. (2)의 층은

$$H^{(l+1)}=\sigma(\tilde D^{-1/2}\tilde A\tilde D^{-1/2}H^{(l)}W^{(l)})$$

입니다. §2.2 Eq. (8)의 `renormalization trick`은 $\tilde A=A+I$를 사용합니다.
여기서는 두 community를 가진 24-node graph와 약한 noisy feature를 만들고, 각 class에서
단 두 노드만 label로 공개합니다. 그래프 이웃 집계가 약한 feature 신호를 매끄럽게 만듭니다.

**성공 기준:** 공개하지 않은 20개 노드의 정확도가 85%보다 커야 합니다.

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** A [N,N], X [N,D] → graph hidden [N,H] → node logits [N,K]
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.manual_seed,np.random.seed,torch.tensor,torch.zeros,Tensor.float,torch.cat,torch.randn,torch.ones,torch.bool,ACCELERATOR.move,torch.allclose,Tensor.sum -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 12개 · 수식 4개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.manual_seed(seed)</code></summary>

#### `torch.manual_seed(seed)`

- **역할:** PyTorch 난수 생성기의 시드를 설정합니다.
- **입력·반환:** 정수 시드를 받고 `Generator`를 반환합니다.
- **이 셀에서 쓰는 이유:** 가중치 초기화와 텐서 샘플링을 반복 가능하게 만듭니다.
- **주의점:** 완전한 결정성에는 알고리즘·장치 설정도 추가로 필요할 수 있습니다.
- **공식 문서:** [torch.manual_seed](https://docs.pytorch.org/docs/stable/generated/torch.manual_seed.html)

</details>

<details>
<summary><code>np.random.seed(seed=None)</code></summary>

#### `np.random.seed(seed=None)`

- **역할:** NumPy의 기존 전역 난수 생성기 상태를 초기화합니다.
- **입력·반환:** 시드를 받고 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** 레거시 `np.random.*` 호출의 수열을 재현합니다.
- **주의점:** 새 코드에서는 독립 상태를 가진 `default_rng`가 권장됩니다.
- **공식 문서:** [np.random.seed](https://numpy.org/doc/stable/reference/random/generated/numpy.random.seed.html)

</details>

<details>
<summary><code>torch.tensor(data, dtype=None, device=None)</code></summary>

#### `torch.tensor(data, dtype=None, device=None)`

- **역할:** 데이터를 복사해 새 PyTorch 텐서를 만듭니다.
- **입력·반환:** 배열형 데이터와 선택적 dtype·device를 받고 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 상수·label·축소 실험 입력을 명시적으로 구성합니다.
- **주의점:** 기존 텐서에서 호출하면 복사가 생기므로 경우에 따라 `as_tensor`가 낫습니다.
- **공식 문서:** [torch.tensor](https://docs.pytorch.org/docs/stable/generated/torch.tensor.html)

</details>

<details>
<summary><code>torch.zeros(*size, dtype=None, device=None)</code></summary>

#### `torch.zeros(*size, dtype=None, device=None)`

- **역할:** 모든 원소가 0인 텐서를 만듭니다.
- **입력·반환:** shape·dtype·device를 받고 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 초기 상태·mask·accumulator를 명시적으로 준비합니다.
- **주의점:** dtype과 device를 생략하면 주변 텐서와 달라질 수 있습니다.
- **공식 문서:** [torch.zeros](https://docs.pytorch.org/docs/stable/generated/torch.zeros.html)

</details>

<details>
<summary><code>tensor.float()</code></summary>

#### `tensor.float()`

- **역할:** 텐서를 32비트 부동소수점 dtype으로 변환합니다.
- **입력·반환:** 텐서를 받고 `torch.float32` 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 정수 입력이나 NumPy 데이터를 신경망 weight와 계산 가능한 dtype으로 맞춥니다.
- **주의점:** class index는 보통 정수 dtype이어야 하므로 모든 텐서에 적용하면 안 됩니다.
- **공식 문서:** [Tensor.float](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.float.html)

</details>

<details>
<summary><code>torch.cat(tensors, dim=0)</code></summary>

#### `torch.cat(tensors, dim=0)`

- **역할:** 기존 축을 따라 텐서를 연결합니다.
- **입력·반환:** 텐서와 축·연산 설정을 받고 변환된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 수식의 축 관계와 batch 연산을 코드로 직접 표현합니다.
- **주의점:** 입력 shape과 broadcasting 규칙을 셀의 shape assertion으로 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
d_{\mathrm{out}}^{(a)}=\sum_{r=1}^{R}d_r^{(a)}
$$

- **기호:** $a$는 연결할 축이고 $R$은 입력 tensor 개수입니다.
- **수식 → 코드:** `dim=a` 축의 길이만 더하고 나머지 축은 그대로 유지합니다.
- **직관:** 이미 존재하는 축을 길게 이어 붙이는 연산입니다.
- **shape 확인:** 연결 축을 제외한 모든 shape과 dtype·device가 같아야 합니다.
- **공식 문서:** [torch.cat](https://docs.pytorch.org/docs/stable/generated/torch.cat.html)

</details>

<details>
<summary><code>torch.randn(*size, generator=None, ...)</code></summary>

#### `torch.randn(*size, generator=None, ...)`

- **역할:** 표준정규분포에서 난수 텐서를 샘플링합니다.
- **입력·반환:** shape와 선택적 생성기를 받고 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 합성 입력·noise·초기 실험값을 만듭니다.
- **주의점:** 재현하려면 해당 PyTorch 생성기의 시드를 먼저 고정해야 합니다.

##### 관련 수식과 코드 연결

$$
X\sim\mathcal N(0,1),\qquad f(x)=\frac{1}{\sqrt{2\pi}}\exp\!\left(-\frac{x^2}{2}\right)
$$

- **기호:** $\mathcal N(0,1)$은 평균 0, 분산 1인 표준정규분포입니다.
- **수식 → 코드:** 요청한 모든 위치에서 독립적인 표준정규 난수를 샘플링합니다.
- **직관:** 학습 noise와 합성 입력의 분포 가정을 코드로 만듭니다.
- **shape 확인:** 인자로 지정한 shape 그대로 반환하며 generator·device가 재현성에 영향을 줍니다.
- **공식 문서:** [torch.randn](https://docs.pytorch.org/docs/stable/generated/torch.randn.html)

</details>

<details>
<summary><code>torch.ones(*size, dtype=None, device=None)</code></summary>

#### `torch.ones(*size, dtype=None, device=None)`

- **역할:** 모든 원소가 1인 텐서를 만듭니다.
- **입력·반환:** shape·dtype·device를 받고 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 초기 weight·mask·스케일을 구성합니다.
- **주의점:** 주변 연산과 dtype·device가 같은지 확인해야 합니다.
- **공식 문서:** [torch.ones](https://docs.pytorch.org/docs/stable/generated/torch.ones.html)

</details>

<details>
<summary><code>torch.bool</code></summary>

#### `torch.bool`

- **역할:** 불리언 dtype
- **입력·반환:** 값이 아니라 텐서 원소의 저장 형식으로 사용됩니다.
- **이 셀에서 쓰는 이유:** mask와 조건 결과를 표현합니다.
- **주의점:** 연산마다 허용 dtype이 다르고 변환 시 정밀도나 메모리가 달라집니다.
- **공식 문서:** [torch.bool](https://docs.pytorch.org/docs/stable/tensor_attributes.html#torch.dtype)

</details>

<details>
<summary><code>ACCELERATOR.move(value)</code></summary>

#### `ACCELERATOR.move(value)`

- **역할:** 텐서나 모듈을 선택된 실행 장치로 옮기는 프로젝트 도우미입니다.
- **입력·반환:** 이동할 값을 받고 같은 논리 값을 장치에 배치해 반환합니다.
- **이 셀에서 쓰는 이유:** 모델과 batch가 서로 다른 장치에 놓이는 오류를 막습니다.
- **주의점:** 컨테이너 재귀 처리 여부와 dtype 변경 여부는 프로젝트 구현을 확인해야 합니다.
- **공식 문서:** 프로젝트 내부 도우미이므로 정의의 docstring을 확인합니다.

</details>

<details>
<summary><code>torch.allclose(input, other, rtol=1e-5, atol=1e-8, equal_nan=False)</code></summary>

#### `torch.allclose(input, other, rtol=1e-5, atol=1e-8, equal_nan=False)`

- **역할:** 두 텐서의 모든 원소가 허용 오차 안에서 가까운지 검사합니다.
- **입력·반환:** 두 텐서와 오차 설정을 받고 Python `bool` 하나를 반환합니다.
- **이 셀에서 쓰는 이유:** 장치나 구현을 바꾼 뒤 수치 결과가 사실상 같은지 확인합니다.
- **주의점:** 정확한 동일성 검사가 아니며 작은 값에서는 `atol`을 직접 정해야 합니다.

##### 관련 수식과 코드 연결

$$
\lvert a_i-b_i\rvert \le \mathrm{atol}+\mathrm{rtol}\,\lvert b_i\rvert
$$

- **기호:** $a_i$는 계산값, $b_i$는 기준값, `atol`은 절대 오차, `rtol`은 상대 오차 허용치입니다.
- **수식 → 코드:** `isclose`는 이 부등식을 원소마다 계산하고, `allclose`는 모든 결과에 논리 AND를 적용합니다.
- **직관:** 완전히 같은 비트인지가 아니라 실험 목적에 충분히 가까운지를 묻는 검사입니다.
- **shape 확인:** 두 입력은 broadcasting 가능해야 합니다. `isclose`는 broadcast shape, `allclose`는 scalar `bool`을 반환합니다.
- **공식 문서:** [torch.allclose](https://docs.pytorch.org/docs/stable/generated/torch.allclose.html)

</details>

<details>
<summary><code>tensor.sum(dim=None, keepdim=False, dtype=None)</code></summary>

#### `tensor.sum(dim=None, keepdim=False, dtype=None)`

- **역할:** 지정 축의 원소 합을 계산합니다.
- **입력·반환:** 텐서와 축 설정을 받고 축약된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 확률 질량, mask 개수, message aggregation을 계산합니다.
- **주의점:** 정수 overflow와 축약 뒤 shape 변화를 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
S=\sum_{i=1}^{N}x_i
$$

- **기호:** $N$은 선택 축의 원소 수입니다.
- **수식 → 코드:** `dim` 또는 `axis`의 모든 원소를 더해 축약합니다.
- **직관:** count, 확률 질량, 이웃 message처럼 누적량을 계산합니다.
- **shape 확인:** 선택한 축은 제거되거나 `keepdim=True`이면 크기 1로 남습니다.
- **공식 문서:** [Tensor.sum](https://docs.pytorch.org/docs/stable/generated/torch.sum.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(150)
np.random.seed(150)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
# This 24-node dense example may be faster with AI_LAB_DEVICE=cpu;
# auto acceleration preserves the same code path for larger graphs.

n_per_class = 12
n_nodes = 2 * n_per_class
labels = torch.tensor([0] * n_per_class + [1] * n_per_class, dtype=torch.long)
adjacency = torch.zeros(n_nodes, n_nodes)

# 각 community 안에서 ring의 1-hop/2-hop 이웃을 연결한다.
for offset in (0, n_per_class):
    for i in range(n_per_class):
        for delta in (1, 2):
            j = offset + (i + delta) % n_per_class
            k = offset + i
            adjacency[k, j] = adjacency[j, k] = 1.0
# 소수의 community 간 edge는 실제 그래프의 noise 역할을 한다.
for left, right in ((2, 14), (8, 20)):
    adjacency[left, right] = adjacency[right, left] = 1.0

feature_rng = torch.Generator().manual_seed(151)
weak_signal = (2 * labels.float() - 1.0)[:, None] * 0.45
features = torch.cat(
    [
        weak_signal + 0.65 * torch.randn(n_nodes, 1, generator=feature_rng),
        0.75 * torch.randn(n_nodes, 2, generator=feature_rng),
        torch.ones(n_nodes, 1),
    ],
    dim=1,
)
train_idx = torch.tensor([0, 5, 12, 17])
test_mask = torch.ones(n_nodes, dtype=torch.bool)
test_mask[train_idx] = False
labels, adjacency, features, train_idx, test_mask = ACCELERATOR.move(
    labels, adjacency, features, train_idx, test_mask
)

assert torch.allclose(adjacency, adjacency.T)
assert features.shape == (24, 4)
print(ACCELERATOR.summary())
edge_count = int(adjacency.sum().item() / 2)
print(f"nodes={n_nodes}, undirected edges={edge_count}, labels used=4")

### 구현 단계 2 · 핵심 연산·모델

- **원문 위치:** §2.2, Eq. (7)–(8)
- **이 셀의 책임:** `normalize_adjacency`의 self-loop와 대칭 degree normalization
- **Tensor shape 계약:** A [N,N], X [N,D] → graph hidden [N,H] → node logits [N,K]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 정규화 행렬의 shape·대칭성·유한값을 검사. 코드의 assertion과 출력으로 바로 확인합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.isfinite -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.isfinite(input)</code></summary>

#### `torch.isfinite(input)`

- **역할:** 텐서의 각 원소가 유한수인지 검사합니다.
- **입력·반환:** 텐서를 받고 같은 shape의 불리언 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** loss나 gradient의 NaN·무한대 발생을 빠르게 찾습니다.
- **주의점:** 검사 결과는 미분 대상이 아니며 원인 자체를 고치지는 않습니다.
- **공식 문서:** [torch.isfinite](https://docs.pytorch.org/docs/stable/generated/torch.isfinite.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
# TODO: A_tilde=A+I와 D_tilde^(-1/2) A_tilde D_tilde^(-1/2)를 구현하세요.
def normalize_adjacency(adjacency):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


a_hat = normalize_adjacency(adjacency)
assert a_hat.shape == adjacency.shape
assert torch.allclose(a_hat, a_hat.T, atol=1e-06)
assert torch.isfinite(a_hat).all()


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

### 구현 단계 3 · 핵심 연산·모델

- **원문 위치:** §2, Eq. (2) / §3.1, Eq. (9)–(10)
- **이 셀의 책임:** `GraphConvolution.forward`의 `A_hat @ X @ W` 전파 / `TinyGCN`의 2-layer propagation과 train mask cross entropy
- **Tensor shape 계약:** A [N,N], X [N,D] → graph hidden [N,H] → node logits [N,K]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** node 수를 보존하는 layer 출력 shape와 finite gradient를 검사 / 미공개 test node 정확도와 학습 loss 감소를 검사. 코드의 assertion과 출력으로 바로 확인합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=nn.Parameter,F.relu,Module.parameters -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 3개 · 수식 1개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>nn.Parameter(data, requires_grad=True)</code></summary>

#### `nn.Parameter(data, requires_grad=True)`

- **역할:** 텐서를 학습 파라미터로 등록합니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.
- **공식 문서:** [nn.Parameter](https://docs.pytorch.org/docs/stable/generated/torch.nn.Parameter.html)

</details>

<details>
<summary><code>F.relu(input, inplace=False)</code></summary>

#### `F.relu(input, inplace=False)`

- **역할:** 음수 activation을 0으로 만드는 ReLU를 적용합니다.
- **입력·반환:** 텐서를 받고 같은 shape의 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 선형 층 사이에 비선형성을 추가합니다.
- **주의점:** 음수 구간 gradient는 0이며 in-place 연산은 autograd 값을 덮어쓸 수 있습니다.

##### 관련 수식과 코드 연결

$$
\operatorname{ReLU}(x)=\max(0,x)
$$

- **기호:** $x$는 선형 변환 뒤 activation입니다.
- **수식 → 코드:** 코드가 음수 원소를 0으로 치환하고 양수는 그대로 통과시킵니다.
- **직관:** 단순하지만 여러 선형 층을 비선형 함수로 조합할 수 있게 합니다.
- **shape 확인:** 입력과 출력 shape은 같습니다.
- **공식 문서:** [F.relu](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.relu.html)

</details>

<details>
<summary><code>model.parameters(recurse=True)</code></summary>

#### `model.parameters(recurse=True)`

- **역할:** 모듈에 등록된 학습 파라미터 iterator를 반환합니다.
- **입력·반환:** 하위 모듈 포함 여부를 받고 `Parameter` iterator를 반환합니다.
- **이 셀에서 쓰는 이유:** optimizer가 갱신할 weight와 bias를 전달합니다.
- **주의점:** iterator는 한 번 소비되며 buffer와 일반 tensor attribute는 포함하지 않습니다.
- **공식 문서:** [Module.parameters](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
# TODO: 논문 식의 W만 정의하세요(bias 없음).
# TODO: A_hat @ X @ W를 반환하세요.
# TODO: 4 -> 12 -> 2 두 층을 정의하세요.
class GraphConvolution(nn.Module):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""

    def __init__(self, in_features, out_features):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        self._exercise_contract_pending = True

    def forward(self, x, normalized_adjacency):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result


class TinyGCN(nn.Module):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""

    def __init__(self):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        self._exercise_contract_pending = True

    def forward(self, x, normalized_adjacency):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result


gcn = TinyGCN().to(DEVICE)
logits = gcn(features, a_hat)
assert logits.shape == (n_nodes, 2)


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

### 구현 단계 4 · 목적함수·학습 update

- **원문 위치:** §3.1, Eq. (9)–(10)
- **이 셀의 책임:** `TinyGCN`의 2-layer propagation과 train mask cross entropy
- **Tensor shape 계약:** A [N,N], X [N,D] → graph hidden [N,H] → node logits [N,K]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** 미공개 test node 정확도와 학습 loss 감소를 검사. 코드의 assertion과 출력으로 바로 확인합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.optim.Adam,torch.no_grad,F.cross_entropy,Optimizer.zero_grad,Tensor.backward,Optimizer.step,Tensor.detach -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 7개 · 수식 4개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.optim.Adam(params, lr=1e-3, ...)</code></summary>

#### `torch.optim.Adam(params, lr=1e-3, ...)`

- **역할:** 1·2차 gradient 모멘트를 추적하는 Adam optimizer를 만듭니다.
- **입력·반환:** 파라미터와 학습률 등을 받고 optimizer 객체를 반환합니다.
- **이 셀에서 쓰는 이유:** 노트북의 trainable parameter를 반복 update합니다.
- **주의점:** weight decay의 해석은 AdamW와 다르며 학습률이 결과에 크게 작용합니다.

##### 관련 수식과 코드 연결

$$
\begin{aligned}m_t&=\beta_1m_{t-1}+(1-\beta_1)g_t,\\v_t&=\beta_2v_{t-1}+(1-\beta_2)g_t^2,\\\theta_t&=\theta_{t-1}-\eta\frac{\hat m_t}{\sqrt{\hat v_t}+\epsilon}\end{aligned}
$$

- **기호:** $g_t$는 gradient, $m_t,v_t$는 1·2차 모멘트, $\eta$는 학습률입니다.
- **수식 → 코드:** `backward()`가 만든 gradient를 읽어 모멘트를 갱신한 뒤 `step()`에서 파라미터에 적용합니다.
- **직관:** 최근 gradient 방향과 크기를 함께 누적해 파라미터별 적응형 보폭을 만듭니다.
- **shape 확인:** 각 trainable parameter와 같은 shape의 모멘트 상태가 optimizer에 저장됩니다.
- **공식 문서:** [torch.optim.Adam](https://docs.pytorch.org/docs/stable/generated/torch.optim.Adam.html)

</details>

<details>
<summary><code>torch.no_grad()</code></summary>

#### `torch.no_grad()`

- **역할:** 블록 안에서 autograd 기록을 끄는 context manager입니다.
- **입력·반환:** 보통 `with`에 사용하며 블록의 텐서는 기본적으로 gradient를 기록하지 않습니다.
- **이 셀에서 쓰는 이유:** 평가나 고정 teacher 추론의 메모리와 연산 부담을 줄입니다.
- **주의점:** `model.eval()`과 역할이 다르므로 평가 모드가 필요하면 둘 다 사용합니다.
- **공식 문서:** [torch.no_grad](https://docs.pytorch.org/docs/stable/generated/torch.no_grad.html)

</details>

<details>
<summary><code>F.cross_entropy(input, target, ...)</code></summary>

#### `F.cross_entropy(input, target, ...)`

- **역할:** logit에 log-softmax와 음의 로그우도를 결합합니다.
- **입력·반환:** 예측·target과 설정을 받고 축약된 loss 또는 같은 shape의 loss를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 objective를 수치화해 역전파 시작점으로 사용합니다.
- **주의점:** 입력 형식, target dtype·shape, `reduction` 의미를 반드시 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\mathcal{L}_{CE}=-\frac{1}{N}\sum_{n=1}^{N}\log\frac{e^{z_{n,y_n}}}{\sum_{c=1}^{C}e^{z_{n,c}}}
$$

- **기호:** $z_{n,c}$는 class logit, $y_n$은 정답 class ID, $C$는 class 수입니다.
- **수식 → 코드:** 함수가 내부에서 `log_softmax`와 정답 class의 음의 로그우도를 결합합니다.
- **직관:** 정답 class 확률이 낮을수록 큰 벌점을 줍니다.
- **shape 확인:** 일반적으로 logits `[N,C,...]`, class-index target `[N,...]`을 사용합니다.
- **공식 문서:** [F.cross_entropy](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.cross_entropy.html)

</details>

<details>
<summary><code>optimizer.zero_grad(set_to_none=True)</code></summary>

#### `optimizer.zero_grad(set_to_none=True)`

- **역할:** 파라미터에 누적된 gradient를 비웁니다.
- **입력·반환:** 초기화 방식을 받고 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** 현재 mini-batch의 gradient만으로 update하도록 학습 단계를 시작합니다.
- **주의점:** 호출하지 않으면 PyTorch gradient가 기본적으로 누적됩니다.
- **공식 문서:** [Optimizer.zero_grad](https://docs.pytorch.org/docs/stable/generated/torch.optim.Optimizer.zero_grad.html)

</details>

<details>
<summary><code>loss.backward()</code></summary>

#### `loss.backward()`

- **역할:** 스칼라 loss에서 계산 그래프를 거슬러 gradient를 계산합니다.
- **입력·반환:** 스칼라 텐서에서는 보통 인자 없이 호출하며 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** optimizer update에 필요한 각 파라미터의 `.grad`를 채웁니다.
- **주의점:** 같은 그래프를 다시 역전파하려면 별도 보존 설정이 필요합니다.

##### 관련 수식과 코드 연결

$$
\frac{\partial L}{\partial x}=\frac{\partial L}{\partial y}\frac{\partial y}{\partial x},\qquad x.\mathrm{grad}\leftarrow x.\mathrm{grad}+\nabla_xL
$$

- **기호:** $L$은 scalar loss이고 $x$는 계산 그래프의 leaf parameter입니다.
- **수식 → 코드:** `backward()`는 연쇄법칙으로 gradient를 계산해 각 parameter의 `.grad`에 누적합니다.
- **직관:** 파라미터를 직접 바꾸는 호출이 아니라 update에 필요한 미분값을 준비하는 호출입니다.
- **shape 확인:** 각 `.grad`는 대응 parameter와 같은 shape이며 scalar가 아닌 출력에는 seed gradient가 필요합니다.
- **공식 문서:** [Tensor.backward](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.backward.html)

</details>

<details>
<summary><code>optimizer.step(closure=None)</code></summary>

#### `optimizer.step(closure=None)`

- **역할:** 현재 gradient와 optimizer 규칙으로 파라미터를 갱신합니다.
- **입력·반환:** 선택적 closure를 받고 optimizer에 따라 loss 또는 `None`을 반환합니다.
- **이 셀에서 쓰는 이유:** 역전파로 계산한 gradient를 실제 학습 update에 반영합니다.
- **주의점:** 일반적으로 `zero_grad`, forward, `backward` 다음에 호출합니다.

##### 관련 수식과 코드 연결

$$
\theta_{t+1}=\mathcal{U}(\theta_t,\nabla_\theta L_t,s_t;\eta)
$$

- **기호:** $\mathcal U$는 optimizer update, $s_t$는 momentum·moment 상태입니다.
- **수식 → 코드:** `step()`은 이미 `.grad`에 저장된 값을 읽어 선택한 optimizer 규칙을 적용합니다.
- **직관:** gradient 계산과 parameter 갱신을 분리하면 학습 lifecycle을 정확히 추적할 수 있습니다.
- **shape 확인:** parameter shape은 유지되고 값과 optimizer state만 바뀝니다.
- **공식 문서:** [Optimizer.step](https://docs.pytorch.org/docs/stable/generated/torch.optim.Optimizer.step.html)

</details>

<details>
<summary><code>tensor.detach()</code></summary>

#### `tensor.detach()`

- **역할:** 같은 저장공간을 공유하면서 현재 계산 그래프에서 분리한 텐서를 만듭니다.
- **입력·반환:** 텐서를 받아 gradient를 추적하지 않는 텐서 view를 반환합니다.
- **이 셀에서 쓰는 이유:** 로그·teacher target·NumPy 변환이 역전파 경로를 만들지 않게 합니다.
- **주의점:** 원본과 저장공간을 공유하므로 in-place 변경은 양쪽에 영향을 줄 수 있습니다.
- **공식 문서:** [Tensor.detach](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.detach.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
# TODO: Eq. (10)처럼 train_idx의 네 label에 대해서만 cross entropy를 계산해 250회 학습하세요.
optimizer = torch.optim.Adam(gcn.parameters(), lr=0.03, weight_decay=5e-4)
loss_history = []
_stage_result = None
assert _stage_result is not None, "TODO: 위 단계의 결과와 검증을 구현하세요."

### 구현 단계 5 · 평가·완료 증거

- **원문 위치:** §3.1, Eq. (9)–(10)
- **이 셀의 책임:** `TinyGCN`의 2-layer propagation과 train mask cross entropy
- **Tensor shape 계약:** A [N,N], X [N,D] → graph hidden [N,H] → node logits [N,K]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** 미공개 test node 정확도와 학습 loss 감소를 검사. 코드의 assertion과 출력으로 바로 확인합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=Module.eval,Tensor.argmax,torch.where -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 3개 · 수식 2개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>model.eval()</code></summary>

#### `model.eval()`

- **역할:** 모듈과 하위 모듈을 평가 모드로 전환합니다.
- **입력·반환:** 인자 없이 호출하고 모듈 자신을 반환합니다.
- **이 셀에서 쓰는 이유:** Dropout을 끄고 BatchNorm의 저장 통계를 사용하게 합니다.
- **주의점:** gradient 기록은 끄지 않으므로 필요하면 `no_grad`도 사용합니다.
- **공식 문서:** [Module.eval](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)

</details>

<details>
<summary><code>tensor.argmax(dim=None, keepdim=False)</code></summary>

#### `tensor.argmax(dim=None, keepdim=False)`

- **역할:** 지정 축에서 최댓값의 index를 찾습니다.
- **입력·반환:** 텐서와 축을 받고 정수 index 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** logit을 예측 class나 선택 action으로 바꿉니다.
- **주의점:** 미분 불가능하며 동점에서는 첫 index가 선택됩니다.

##### 관련 수식과 코드 연결

$$
k^\star=\operatorname*{arg\,max}_{k}x_k
$$

- **기호:** $k^\star$는 가장 큰 값 자체가 아니라 그 값의 index입니다.
- **수식 → 코드:** `dim` 또는 `axis`가 어느 후보 집합에서 최댓값 index를 고를지 정합니다.
- **직관:** 연속 score를 이산 class·action 결정으로 바꾸는 선택 연산입니다.
- **shape 확인:** 선택 축이 제거되며 미분 가능한 gradient 경로는 만들지 않습니다.
- **공식 문서:** [Tensor.argmax](https://docs.pytorch.org/docs/stable/generated/torch.argmax.html)

</details>

<details>
<summary><code>torch.where(condition, input, other)</code></summary>

#### `torch.where(condition, input, other)`

- **역할:** 조건에 따라 두 텐서의 원소를 선택합니다.
- **입력·반환:** 불리언 조건과 두 입력을 받고 broadcast된 결과를 반환합니다.
- **이 셀에서 쓰는 이유:** mask 기반 분기와 piecewise 수식을 벡터화합니다.
- **주의점:** 세 입력의 broadcasting과 두 branch의 dtype·device를 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
y_i=\begin{cases}x_i,&c_i=\mathrm{True}\\z_i,&c_i=\mathrm{False}\end{cases}
$$

- **기호:** $c_i$는 조건, $x_i,z_i$는 두 후보 값입니다.
- **수식 → 코드:** API가 Python 분기 대신 원소별로 두 tensor 중 하나를 선택합니다.
- **직관:** piecewise 수식과 mask 기반 update를 병렬 tensor 연산으로 표현합니다.
- **shape 확인:** 세 입력을 broadcasting한 공통 shape이 출력 shape입니다.
- **공식 문서:** [torch.where](https://docs.pytorch.org/docs/stable/generated/torch.where.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
# TODO: train/test accuracy를 계산하고 true/predicted graph 및 loss를 그리세요.
def draw_graph(axis, colors, title):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


_stage_result = None
assert _stage_result is not None, "TODO: 위 단계의 결과와 검증을 구현하세요."


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

## 7. 해석 TODO

1. self-loop를 제거하면 한 층에서 자기 feature는 어떻게 되나요?
2. loss에 `test_mask`를 포함하면 왜 semi-supervised 실험이 아니게 되나요?
3. 이 dense 구현과 논문의 citation graph 구현 사이 계산 복잡도 차이를 쓰세요.

## 마무리 관찰 기록

아래 세 문장을 직접 완성하세요.

1. 이 논문의 핵심 연산은 `_____`이고, 코드에서는 `_____`에 해당한다.
2. 원 규모 실험 대신 미니 재현을 사용했기 때문에 확인할 수 없는 주장은 `_____`이다.
3. 한 가지 변수를 바꾸어 다시 실행한다면 `_____`를 바꾸고 `_____`를 측정하겠다.

**추가 실험:** seed는 유지한 채 한 변수만 바꾸고, 변경 전후의 metric을 표로 남기세요.